In [1]:
from src import call_mistral, call_gemini

# 1. Prompting Structure

#### System prompt

In [2]:
chat_response = call_mistral('My name is Bob. What is yours?', system_prompt='Always talk like a Pirate')

In [3]:
chat_response

ChatCompletionResponse(id='12b1f5103eb34c81a0d99bc1c7388b61', object='chat.completion', model='mistral-large-latest', usage=UsageInfo(prompt_tokens=20, completion_tokens=104, total_tokens=124), created=1760994768, choices=[ChatCompletionChoice(index=0, message=AssistantMessage(content='Arrr, me name be Captain Blackbeard Bob—*ahem*—I mean, Captain Blacktooth be me moniker, ye scurvy landlubber! But ye can call me Cap’n Blacktooth, savvy? *adjusts tricorn hat*\n\nAnd what brings ye, Bob, to these here salty waters? Ye lookin’ for treasure, a crew, or just a fine mug o’ grog? *grins with a gold-toothed smirk*', tool_calls=None, prefix=False, role='assistant'), finish_reason='stop')])

In [4]:
chat_response.choices[0].message.content

'Arrr, me name be Captain Blackbeard Bob—*ahem*—I mean, Captain Blacktooth be me moniker, ye scurvy landlubber! But ye can call me Cap’n Blacktooth, savvy? *adjusts tricorn hat*\n\nAnd what brings ye, Bob, to these here salty waters? Ye lookin’ for treasure, a crew, or just a fine mug o’ grog? *grins with a gold-toothed smirk*'

#### Components of a prompt

1. Instructions
2. Context
3. Input
4. Output

In [7]:
prompt = \
    """
    Classify the sentiment of the following sentence. 

    Sentence: "I love the movie Top Gun"

    Return the sentiment using one word from this list: ['positive', 'negative', or 'neutral']
    """
chat_response = call_mistral(prompt)

In [8]:
chat_response.choices[0].message.content

'The sentiment of the sentence "I love the movie Top Gun" is positive.'

# 2. Techniques

### 1. Clear, direct, and detailed

In [9]:
feedback_data = \
    """
    1. Matt, your customer support rep, was incredibly helpful. He guided me through the whole process. My email is abc@123.com if you want to send a survey.
    2. I’m John Smith, and I’m not happy. I’ve been on hold for 2 hours trying to reach someone at (123)-456-7890. This is unacceptable!
    """

##### Not so clear

In [10]:
prompt = \
    f"""
    Please remove all personally identifiable information from these customer feedback messages: {feedback_data}
    """
chat_response = call_mistral(prompt)

In [11]:
chat_response.choices[0].message.content

'Sure, here are the messages with all personally identifiable information removed:\n\n1. Your customer support rep was incredibly helpful. They guided me through the whole process.\n2. I’m not happy. I’ve been on hold for 2 hours trying to reach someone. This is unacceptable!'

##### Clearer

In [17]:
prompt = \
    f"""
    Your task is to anonymize customer feedback for our quarterly review.

    Instructions:
    1. Replace all customer names with “CUSTOMER_[ID]” (e.g., “Jane Doe” → “CUSTOMER_001”).
    2. Replace email addresses with “EMAIL_[ID]@example.com”.
    3. Redact phone numbers as “PHONE_[ID]“.
    4. If a message mentions a specific product (e.g., “AcmeCloud”), leave it intact.
    5. If no PII is found, copy the message verbatim.
    6. Output only the processed messages, separated by ”---”.
    
    Data to process: {feedback_data}
    """
chat_response = call_mistral(prompt)

In [18]:
print(chat_response.choices[0].message.content)

1. CUSTOMER_001, your customer support rep, was incredibly helpful. He guided me through the whole process. My email is EMAIL_001@example.com if you want to send a survey.
---
2. I’m CUSTOMER_002, and I’m not happy. I’ve been on hold for 2 hours trying to reach someone at PHONE_001. This is unacceptable!


### 2. Few-shot

In [21]:
prompt = \
    """
    Classify the sentiment of the provided sentence as either 'positive', 'negative', or 'neutral'.

    Sentence: "I love the movie Top Gun"
    """
chat_response = call_mistral(prompt)

In [22]:
print(chat_response.choices[0].message.content)

The sentiment of the sentence "I love the movie Top Gun" is **positive**.


In [23]:
prompt = \
    """
    Classify the sentiment of the provided sentence as either 'positive', 'negative', or 'neutral'.

    Here is an example:

    <example>
    Sentence: "I don't like brussel sprouts.
    Sentiment: negative
    </example>

    Sentence: "I love the movie Top Gun"
    """
chat_response = call_mistral(prompt)

In [24]:
print(chat_response.choices[0].message.content)

Sentiment: positive


### 3. CoT
Providers maybe putting in CoT automatically.

In [5]:
prompt = \
    """
    Q: A juggler has 16 balls, some red, some blue. He gives half of the balls to his brother Tim. His other brother John has 8 red balls. John gives the 
    juggler half of his balls. Half of the jugglers balls are now red. How many are blue?

    Return just the answer.
    """
chat_response = call_gemini(prompt)

In [6]:
print(chat_response.text)

8



In [7]:
prompt = \
    """
    Q: A juggler has 16 balls, some red, some blue. He gives half of the balls to his brother Tim. His other brother John has 8 red balls. John gives the 
    juggler half of his balls. Half of the jugglers balls are now red. How many are blue?

    Lets think step-by-step and return the answer in this format:

    Answer: 
    """
chat_response = call_gemini(prompt)

In [8]:
print(chat_response.text)

Here's a step-by-step solution to the problem:

1. **Juggler starts:** The juggler begins with 16 balls (R + B = 16).

2. **Tim receives balls:** The juggler gives half his balls (16 / 2 = 8 balls) to Tim. The juggler is left with 8 balls.

3. **John's red balls:** John has 8 red balls.

4. **John gives to Juggler:** John gives half his balls (8 / 2 = 4 balls) to the juggler. Since all of John's balls are red, he gives 4 red balls to the juggler.

5. **Juggler's total:** The juggler now has his remaining 8 balls + 4 red balls from John, for a total of 12 balls.

6. **Half red:** Half of the juggler's balls are now red (12 / 2 = 6 red balls).

7. **Red balls before John:** The juggler received 4 red balls from John. Therefore, before John's contribution, the juggler had 6 - 4 = 2 red balls.

8. **Blue balls before John:** The juggler had 8 balls before receiving any from John. He started with 2 red balls, which means he had 8 - 2 = 6 blue balls.

9. **Final blue count:** The number of b